<a href="https://colab.research.google.com/github/Williamzamo/Trabajos-algebra-lineal/blob/main/Interpolacion_polinomial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto computacional: Interpolación polinomial

**Curso:** Álgebra Lineal II — Programa de Matemáticas Aplicadas y Computacionales, UMNG
**Profesor:** Juan Camilo Torres Chaves

Este notebook cubre:

1. **Parte 1:** dado un conjunto de $n+1$ puntos con abscisas distintas, hallar el único polinomio de
   grado a lo más $n$ que pasa por todos ellos, usando la matriz de Vandermonde.
2. **Parte 2:** aplicar esa idea para hallar fórmulas explícitas de sumas de potencias
   $S_d(n) = \sum_{k=1}^{n} k^d$.


## 1. Teoría: interpolación polinomial y la matriz de Vandermonde

Dos puntos distintos $(x_0,y_0)$, $(x_1,y_1)$ con $x_0\neq x_1$ determinan una única recta
$f(x)=ax+b$. Los coeficientes se obtienen resolviendo

$$
\begin{pmatrix}1 & x_0\\ 1 & x_1\end{pmatrix}
\begin{pmatrix}b\\ a\end{pmatrix}
=
\begin{pmatrix}y_0\\ y_1\end{pmatrix}.
$$

Como $\det = x_1-x_0 \neq 0$, el sistema tiene solución única.

**Caso general.** Dados $n+1$ puntos $(x_0,y_0),\dots,(x_n,y_n)$ con todas las abscisas distintas, existe
un único polinomio $f(x) = a_n x^n + \cdots + a_1 x + a_0$ de grado a lo más $n$ tal que
$f(x_i) = y_i$ para todo $i$. Esto equivale al sistema lineal

$$
\begin{pmatrix}
1 & x_0 & x_0^2 & \cdots & x_0^n\\
1 & x_1 & x_1^2 & \cdots & x_1^n\\
\vdots & \vdots & \vdots & \ddots & \vdots\\
1 & x_n & x_n^2 & \cdots & x_n^n
\end{pmatrix}
\begin{pmatrix}a_0\\ a_1\\ \vdots\\ a_n\end{pmatrix}
=
\begin{pmatrix}y_0\\ y_1\\ \vdots\\ y_n\end{pmatrix}.
$$

La matriz de coeficientes se llama **matriz de Vandermonde**, y es invertible exactamente cuando
$x_0,\dots,x_n$ son todos distintos.


### Teorema 1

La matriz de Vandermonde asociada a $x_0,\dots,x_n$ es invertible si y solo si estos valores son todos
diferentes.



### Teorema 2

$$
\det\begin{pmatrix}
1 & x_0 & x_0^2 & \cdots & x_0^n\\
1 & x_1 & x_1^2 & \cdots & x_1^n\\
\vdots & & & \ddots & \vdots\\
1 & x_n & x_n^2 & \cdots & x_n^n
\end{pmatrix}
=\prod_{0\le i<j\le n}(x_j-x_i).
$$



### Ejemplo de referencia (el mismo que usaremos en el código)

Hallar el polinomio de grado a lo más $4$ que interpola los puntos
$(0,1),\,(1,2),\,(2,0),\,(3,-1),\,(4,2)$:

$$
\begin{pmatrix}a_0\\a_1\\a_2\\a_3\\a_4\end{pmatrix}
=
\begin{pmatrix}
1&0&0&0&0\\
1&1&1&1&1\\
1&2&4&8&16\\
1&3&9&27&81\\
1&4&16&64&256
\end{pmatrix}^{-1}
\begin{pmatrix}1\\2\\0\\-1\\2\end{pmatrix}
=
\begin{pmatrix}1\\ 49/12\\ -95/24\\ 11/12\\ -1/24\end{pmatrix}.
$$

Es decir,

$$f(x) = -\tfrac{1}{24}x^4 + \tfrac{11}{12}x^3 - \tfrac{95}{24}x^2 + \tfrac{49}{12}x + 1.$$

Verificaremos este resultado numéricamente con el código de la siguiente sección.


## 1.1 Resolviendo el sistema sin calcular $V^{-1}$: factorización LU

En la teoría escribimos $a = V^{-1}y$, pero eso no es lo que haremos aquí: en vez de invertir $V$, la
factorizamos como $V = LU$, con $L$ triangular inferior (unos en la diagonal) y $U$ triangular superior.
Con $L$ y $U$ ya calculadas, resolvemos $Va=y$ en dos pasos, sin invertir nada.

Trabajamos con el mismo ejemplo de referencia: los puntos $(0,1),(1,2),(2,0),(3,-1),(4,2)$, con

$$V=\begin{pmatrix}
1&0&0&0&0\\
1&1&1&1&1\\
1&2&4&8&16\\
1&3&9&27&81\\
1&4&16&64&256
\end{pmatrix},
\qquad
\vec y=\begin{pmatrix}1\\2\\0\\-1\\2\end{pmatrix}.$$

Factorización $V=LU$:

$$L=\begin{pmatrix}
1&0&0&0&0\\
1&1&0&0&0\\
1&2&1&0&0\\
1&3&3&1&0\\
1&4&6&4&1
\end{pmatrix},
\qquad
U=\begin{pmatrix}
1&0&0&0&0\\
0&1&1&1&1\\
0&0&2&6&14\\
0&0&0&6&36\\
0&0&0&0&24
\end{pmatrix}.$$



Sustitución hacia adelante. resolver $L\vec z=\vec y$.

$$z_0 = 1$$
$$z_0+z_1=2 \;\Rightarrow\; z_1=1$$
$$z_0+2z_1+z_2=0 \;\Rightarrow\; 1+2+z_2=0 \;\Rightarrow\; z_2=-3$$
$$z_0+3z_1+3z_2+z_3=-1 \;\Rightarrow\; 1+3-9+z_3=-1 \;\Rightarrow\; z_3=4$$
$$z_0+4z_1+6z_2+4z_3+z_4=2 \;\Rightarrow\; 1+4-18+16+z_4=2 \;\Rightarrow\; z_4=-1$$

Entonces $\vec z=(1,\,1,\,-3,\,4,\,-1)$.

Sustitución hacia atrás. resolver $U\vec a=\vec z$.

$$24\,a_4=-1 \;\Rightarrow\; a_4=-\tfrac{1}{24}$$
$$6a_3+36a_4=4 \;\Rightarrow\; 6a_3-\tfrac{36}{24}=4 \;\Rightarrow\; a_3=\tfrac{11}{12}$$
$$2a_2+6a_3+14a_4=-3 \;\Rightarrow\; 2a_2+\tfrac{11}{2}-\tfrac{7}{12}=-3 \;\Rightarrow\; a_2=-\tfrac{95}{24}$$
$$a_1+a_2+a_3+a_4=1 \;\Rightarrow\; a_1-\tfrac{95}{24}+\tfrac{11}{12}-\tfrac{1}{24}=1 \;\Rightarrow\; a_1=\tfrac{49}{12}$$
$$a_0=1$$

$$\boxed{\vec a = \left(1,\ \tfrac{49}{12},\ -\tfrac{95}{24},\ \tfrac{11}{12},\ -\tfrac{1}{24}\right)}$$

Este es el mismo resultado que obtuvimos antes (con la inversa) y el mismo que da `np.linalg.solve` por eso en nuestro programa no tenemos la necesidad de calcular la inversa


## 2. Parte 1: implementación

**Estrategia del código:**

1. Construir la matriz de Vandermonde a partir de las abscisas $x_i$.
2. Resolver el sistema lineal $V\,a = y$ para obtener los coeficientes $a_0,\dots,a_n$
   (usamos `np.linalg.solve`, que es numéricamente más estable que invertir $V$ explícitamente,
   aunque matemáticamente es equivalente a lo mostrado en la teoría).
3. Formatear el resultado como un polinomio legible.


In [1]:
import numpy as np

# Los mismos puntos usados en el Ejemplo 3 de la guía teórica, modifiquelos o borrelos y agrege unos nuevos para provar mas casos.
puntos = [(0, 1), (1, 2), (2, 0), (3, -1), (4, 2)]

def matriz_vandermonde(xs):
    '''Construye la matriz de Vandermonde.'''
    n = len(xs) - 1
    filas = []
    for x in xs:
        fila = [x ** potencia for potencia in range(n + 1)]
        filas.append(fila)
    return filas

def interpolar(puntos):
    xs = [p[0] for p in puntos]
    ys = [p[1] for p in puntos]
    if len(set(xs)) != len(xs):
        raise ValueError("Las coordenadas x deben ser todas distintas.")
    V = np.array(matriz_vandermonde(xs), dtype=float)
    print("Matriz de Vandermonde V:")
    print(V)
    y = np.array(ys, dtype=float)
    print("\nVector y:", y)
    # Resolvemos V @ a = y (equivalente a a = V^{-1} y, pero mas estable numericamente)
    coeficientes = np.linalg.solve(V, y)
    return coeficientes

def formatear_polinomio(coeficientes):
    '''Convierte el vector de coeficientes en un string tipo 'a0 + a1 x + a2 x^2 + ...' su funcion es puramente estetica'''
    terminos = []
    for i, a in enumerate(coeficientes):
        a_redondeado = round(a, 4)
        if a_redondeado == 0:
            continue
        if i == 0:
            terminos.append(f"{a_redondeado}")
        elif i == 1:
            terminos.append(f"{a_redondeado}x")
        else:
            terminos.append(f"{a_redondeado}x^{i}")
    return " + ".join(terminos) if terminos else "0"

coeficientes = interpolar(puntos)
print("\nPuntos a interpolar:", puntos)
print("\nCoeficientes [a0, a1, a2, a3, a4]:")
print(coeficientes)
print("\nPolinomio interpolador:")
print("f(x) =", formatear_polinomio(coeficientes))


Matriz de Vandermonde V:
[[  1.   0.   0.   0.   0.]
 [  1.   1.   1.   1.   1.]
 [  1.   2.   4.   8.  16.]
 [  1.   3.   9.  27.  81.]
 [  1.   4.  16.  64. 256.]]

Vector y: [ 1.  2.  0. -1.  2.]

Puntos a interpolar: [(0, 1), (1, 2), (2, 0), (3, -1), (4, 2)]

Coeficientes [a0, a1, a2, a3, a4]:
[ 1.          4.08333333 -3.95833333  0.91666667 -0.04166667]

Polinomio interpolador:
f(x) = 1.0 + 4.0833x + -3.9583x^2 + 0.9167x^3 + -0.0417x^4


## 3. Teoría: sumas de potencias $S_d(n)$

Definimos $S_d(n) = \sum_{k=1}^{n} k^d$. Se conocen fórmulas clásicas como

$$
S_0(n) = n,\qquad
S_1(n) = \frac{n(n+1)}{2},\qquad
S_2(n) = \frac{n(n+1)(2n+1)}{6},\qquad
S_3(n) = \left(\frac{n(n+1)}{2}\right)^2.
$$

En todos los casos $S_d(n)$ resulta ser un **polinomio en $n$ de grado $d+1$**:

### Teorema 4

$S_d(n)$ es un polinomio de grado $d+1$ en la variable $n$.

### Consecuencia práctica

Como $S_d(n)$ es un polinomio de grado $d+1$, basta conocer su valor en $d+2$ puntos para determinarlo
exactamente por interpolación (usando la función `interpolar` de la Parte 1). Una elección natural
de puntos es $n = 0, 1, 2, \dots, d+1$, calculando cada $S_d(n)$ por suma directa.


### Ejemplo de referencia: $S_2(n)$

Como $S_2(n)$ es de grado $3$, usamos $4$ puntos: $S_2(0)=0,\ S_2(1)=1,\ S_2(2)=5,\ S_2(3)=14$.

$$
\begin{pmatrix}a_0\\a_1\\a_2\\a_3\end{pmatrix}
=
\begin{pmatrix}
1&0&0&0\\
1&1&1&1\\
1&2&4&8\\
1&3&9&27
\end{pmatrix}^{-1}
\begin{pmatrix}0\\1\\5\\14\end{pmatrix}
=
\begin{pmatrix}0\\ 1/6\\ 1/2\\ 1/3\end{pmatrix},
$$

es decir $S_2(n) = \tfrac{1}{3}n^3+\tfrac{1}{2}n^2+\tfrac{1}{6}n = \dfrac{n(n+1)(2n+1)}{6}$, que coincide
con la fórmula clásica.


## 4. Parte 2: implementación

**Estrategia del código:**

1. Fijar un grado $d$.
2. Calcular $S_d(n)$ por suma directa para $n = 0,1,\dots,d+1$ (esto da $d+2$ puntos).
3. Interpolar esos puntos con la función `interpolar` ya construida — el resultado son los coeficientes
   del polinomio $S_d(n)$.
4. Mostrar la fórmula resultante.


In [ ]:
D = 2  # Cambia este valor para hallar la formula de S_d(n) con otro d

def suma_potencia_directa(d, n):
    '''Calcula S_d(n) = sum_{k=0}^{n} k^d sumando termino a termino (fuerza bruta).'''
    return sum(k ** d for k in range(n + 1))

def construir_puntos(d):
    '''Genera d+2 puntos (n, S_d(n)) para n = 0, ..., d+1, suficientes para
    determinar el polinomio de grado d+1 por interpolacion.'''
    puntos = []
    for n in range(d + 2):
        puntos.append((n, suma_potencia_directa(d, n)))
    return puntos

def encontrar_formula(d):
    puntos_sd = construir_puntos(d)
    coeficientes = interpolar(puntos_sd)
    return puntos_sd, coeficientes

def mostrar_formula(d):
    print("=" * 60)
    print(f"Hallando la formula para S_{d}(n) = sum_{{k=1}}^n k^{d}")
    print("=" * 60)

    puntos_sd, coeficientes = encontrar_formula(d)

    print(f"\nPuntos usados para interpolar (d+2 = {d + 2} puntos):")
    print(puntos_sd)

    print(f"\nCoeficientes [a_0, ..., a_{d+1}]:")
    print(coeficientes)

    print(f"\nFormula obtenida:")
    print(f"S_{d}(n) =", formatear_polinomio(coeficientes))
    print()

mostrar_formula(D)


## 5. Programa final

Versión final consolidada


In [ ]:
import numpy as np

puntos = [(0, 1), (1, 2), (2, 0), (3, -1), (4, 2)]

def matriz_vandermonde(xs):
    n = len(xs) - 1
    filas = []
    for x in xs:
        fila = [x ** potencia for potencia in range(n + 1)]
        filas.append(fila)
    return filas

def interpolar(puntos):
    xs = [p[0] for p in puntos]
    ys = [p[1] for p in puntos]
    if len(set(xs)) != len(xs):
        raise ValueError("Las coordenadas x deben ser todas distintas.")
    V = np.array(matriz_vandermonde(xs), dtype=float)
    y = np.array(ys, dtype=float)
    coeficientes = np.linalg.solve(V, y)
    return coeficientes

def formatear_polinomio(coeficientes):
    terminos = []
    for i, a in enumerate(coeficientes):
        a_redondeado = round(a, 4)
        if a_redondeado == 0:
            continue
        if i == 0:
            terminos.append(f"{a_redondeado}")
        elif i == 1:
            terminos.append(f"{a_redondeado}x")
        else:
            terminos.append(f"{a_redondeado}x^{i}")
    return " + ".join(terminos) if terminos else "0"

def main():
    coeficientes = interpolar(puntos)
    print("Puntos a interpolar:", puntos)
    print("\nCoeficientes:")
    print(coeficientes)
    print("\nPolinomio:")
    print("f(x) =", formatear_polinomio(coeficientes))

main()

# -----------------------------------parte 2-----------------------------------------------------
D = 4  # Cambia este valor para hallar la formula de S_d(n) con otro d

def suma_potencia_directa(d, n):
    return sum(k ** d for k in range(n + 1))

def construir_puntos(d):
    puntos = []
    for n in range(d + 2):
        puntos.append((n, suma_potencia_directa(d, n)))
    return puntos

def encontrar_formula(d):
    puntos_sd = construir_puntos(d)
    coeficientes = interpolar(puntos_sd)
    return puntos_sd, coeficientes

def mostrar_formula(d):
    print("=" * 60)
    print(f"Hallando la formula para S_{d}(n) = sum_{{k=1}}^n k^{d}")
    print("=" * 60)

    puntos_sd, coeficientes = encontrar_formula(d)

    print(f"\nPuntos usados para interpolar (d+2 = {d + 2} puntos):")
    print(puntos_sd)

    print(f"\nCoeficientes [a_0, ..., a_{d+1}]:")
    print(coeficientes)

    print(f"\nFormula obtenida:")
    print(f"S_{d}(n) =", formatear_polinomio(coeficientes))
    print()

def main_sumas():
    mostrar_formula(D)

main_sumas()
